# Carelon App — Entitlement Tables Setup

Creates three Delta tables for the Carelon App permission system:

1. **`permission_types`** — Reference table defining what permissions exist per type (files, jobs, etc.)  
2. **`permission_assignments`** — Stores who (user/group/SP) has which permission on which resource path
3. **`volume_grants`** — Tracks UC-level GRANT/REVOKE state per entity + volume

The UI renders dynamically from `permission_types`, so adding a new type+action here automatically shows it in the admin UI.

**Configuration:** Set catalog, schema, volume, and SP UUID in the Parameters cell below.

In [0]:
# ======== Configuration Parameters ========
# Change these values to deploy to a different catalog/schema/volume

CATALOG = "aw_serverless_stable_catalog"
SCHEMA = "carelon"
VOLUME = "dxutility"

# Service Principal application ID (UUID) for UC GRANTs
# This is the SP that the Carelon App runs as
APP_SP_UUID = "f88b1e5f-61ff-418a-85b5-31547db575af"

# Derived fully-qualified names
PERM_TYPES_TABLE = f"{CATALOG}.{SCHEMA}.permission_types"
PERM_ASSIGNMENTS_TABLE = f"{CATALOG}.{SCHEMA}.permission_assignments"
VOLUME_GRANTS_TABLE = f"{CATALOG}.{SCHEMA}.volume_grants"
VOLUME_FQN = f"{CATALOG}.{SCHEMA}.{VOLUME}"

print(f"Catalog:              {CATALOG}")
print(f"Schema:               {SCHEMA}")
print(f"Volume:               {VOLUME}")
print(f"SP UUID:              {APP_SP_UUID}")
print(f"Permission Types:     {PERM_TYPES_TABLE}")
print(f"Permission Assigns:   {PERM_ASSIGNMENTS_TABLE}")
print(f"Volume Grants:        {VOLUME_GRANTS_TABLE}")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {PERM_TYPES_TABLE} (
  permission_type STRING NOT NULL COMMENT 'Category: files, jobs, clusters, etc.',
  action STRING NOT NULL COMMENT 'Specific permission action within the type',
  display_name STRING COMMENT 'Human-readable label for the UI',
  description STRING COMMENT 'Tooltip/help text for the action',
  sort_order INT DEFAULT 0 COMMENT 'Display order within the type',
  is_active BOOLEAN DEFAULT TRUE COMMENT 'Soft-disable without deleting',
  created_at TIMESTAMP DEFAULT current_timestamp(),
  updated_at TIMESTAMP DEFAULT current_timestamp()
)
USING DELTA
COMMENT 'Reference table defining available permission types and actions for the Carelon App'
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true', 'delta.feature.allowColumnDefaults' = 'supported')
""")
print(f"Created: {PERM_TYPES_TABLE}")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {PERM_ASSIGNMENTS_TABLE} (
  assignment_id BIGINT GENERATED ALWAYS AS IDENTITY COMMENT 'Auto-increment PK',
  permission_type STRING NOT NULL COMMENT 'FK to permission_types.permission_type',
  action STRING NOT NULL COMMENT 'FK to permission_types.action',
  entity_id STRING NOT NULL COMMENT 'Databricks SCIM ID of the user/group/SP',
  entity_type STRING NOT NULL COMMENT 'user, group, or service_principal',
  entity_name STRING COMMENT 'Email (users) or display name (groups) - used for matching',
  assigned_by STRING COMMENT 'Email of the admin who assigned this',
  assigned_at TIMESTAMP DEFAULT current_timestamp(),
  is_active BOOLEAN DEFAULT TRUE COMMENT 'Soft-revoke without deleting',
  resource_path STRING COMMENT 'Volume folder path this permission applies to (NULL = all folders)'
)
USING DELTA
COMMENT 'Stores permission assignments: who has what permission on which resource path'
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true', 'delta.feature.allowColumnDefaults' = 'supported')
""")
print(f"Created: {PERM_ASSIGNMENTS_TABLE}")

In [0]:
# Grant the app's Service Principal access to all entitlement tables + volume
grants = [
    f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `{APP_SP_UUID}`",
    f"GRANT USE SCHEMA ON SCHEMA {CATALOG}.{SCHEMA} TO `{APP_SP_UUID}`",
    f"GRANT SELECT, MODIFY ON TABLE {PERM_TYPES_TABLE} TO `{APP_SP_UUID}`",
    f"GRANT SELECT, MODIFY ON TABLE {PERM_ASSIGNMENTS_TABLE} TO `{APP_SP_UUID}`",
    f"GRANT SELECT, MODIFY ON TABLE {VOLUME_GRANTS_TABLE} TO `{APP_SP_UUID}`",
    # SP needs MANAGE on volume to execute GRANT/REVOKE on behalf of admins
    f"GRANT MANAGE ON VOLUME {VOLUME_FQN} TO `{APP_SP_UUID}`",
]

for stmt in grants:
    print(f"  Executing: {stmt}")
    spark.sql(stmt)

print("\n✓ All grants applied successfully.")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {VOLUME_GRANTS_TABLE} (
  grant_id BIGINT GENERATED ALWAYS AS IDENTITY COMMENT 'Auto-increment PK',
  entity_id STRING NOT NULL COMMENT 'Databricks SCIM ID of user/group/SP',
  entity_type STRING NOT NULL COMMENT 'user, group, or service_principal',
  entity_name STRING NOT NULL COMMENT 'Email (users) or displayName (groups/SPs) - used in GRANT SQL',
  volume_fqn STRING NOT NULL COMMENT 'Fully qualified volume: catalog.schema.volume',
  uc_privilege STRING NOT NULL COMMENT 'READ_VOLUME, WRITE_VOLUME, or MANAGE',
  granted_by STRING COMMENT 'Admin who triggered the grant',
  granted_at TIMESTAMP DEFAULT current_timestamp(),
  revoked_at TIMESTAMP COMMENT 'When revoked (NULL if active)',
  is_active BOOLEAN DEFAULT TRUE COMMENT 'FALSE after REVOKE executed'
)
USING DELTA
COMMENT 'Tracks Unity Catalog volume-level grants applied by the Carelon App permissions UI'
TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true', 'delta.feature.allowColumnDefaults' = 'supported')
""")
print(f"✓ Created: {VOLUME_GRANTS_TABLE}")

In [0]:
# Seed the reference table with files and jobs permission types
spark.sql(f"""
MERGE INTO {PERM_TYPES_TABLE} AS target
USING (
  SELECT * FROM VALUES
    -- Files permissions
    ('files', 'browse', 'Browse', 'View list of files in Volumes', 10, TRUE),
    ('files', 'upload', 'Upload', 'Upload data files and trigger tokenization', 20, TRUE),
    ('files', 'download', 'Download', 'Download files from Volumes', 30, TRUE),
    ('files', 'delete', 'Delete', 'Remove files from Volumes', 40, TRUE),
    ('files', 'preview', 'Preview', 'View first N rows of a file', 50, TRUE),
    ('files', 'detokenize', 'Detokenize', 'Reverse tokenization (download only)', 60, TRUE),
    ('files', 'share', 'Share', 'Share files via Delta Sharing', 70, TRUE),
    ('files', 'manage_permissions', 'Manage Permissions', 'Assign/revoke permissions for others', 80, TRUE),
    -- Jobs permissions
    ('jobs', 'view_jobs', 'View Jobs', 'List and view job details', 10, TRUE),
    ('jobs', 'create_jobs', 'Create Jobs', 'Create new Databricks jobs', 20, TRUE),
    ('jobs', 'run_jobs', 'Run / Trigger Jobs', 'Manually trigger job runs', 30, TRUE),
    ('jobs', 'cancel_runs', 'Cancel Runs', 'Cancel running or queued job runs', 40, TRUE),
    ('jobs', 'manage_jobs', 'Manage Jobs (edit/delete)', 'Edit job settings or delete jobs', 50, TRUE)
  AS src(permission_type, action, display_name, description, sort_order, is_active)
) AS source
ON target.permission_type = source.permission_type AND target.action = source.action
WHEN NOT MATCHED THEN
  INSERT (permission_type, action, display_name, description, sort_order, is_active)
  VALUES (source.permission_type, source.action, source.display_name, source.description, source.sort_order, source.is_active)
""")
print("✓ Seeded permission_types with files + jobs actions")

In [0]:
# Verify all tables and seed data
print("=== Permission Types ===")
display(spark.sql(f"SELECT permission_type, action, display_name, description, sort_order FROM {PERM_TYPES_TABLE} WHERE is_active = TRUE ORDER BY permission_type, sort_order"))

print(f"\n=== {PERM_ASSIGNMENTS_TABLE} schema ===")
spark.sql(f"DESCRIBE TABLE {PERM_ASSIGNMENTS_TABLE}").show(truncate=False)

print(f"\n=== {VOLUME_GRANTS_TABLE} schema ===")
spark.sql(f"DESCRIBE TABLE {VOLUME_GRANTS_TABLE}").show(truncate=False)

## Schema Design — UC Volume Privilege Integration

### Architecture Overview

The permissions system operates at **two levels**:

1. **Unity Catalog Level** — `GRANT READ VOLUME / WRITE VOLUME / MANAGE ON VOLUME` controls access to the physical volume
2. **App Level** — `permission_assignments` + `resource_path` controls which folders within the volume a user can access and what actions they can perform

### Action → UC Privilege Mapping

| UI Action | UC Privilege | Notes |
|-----------|-------------|-------|
| Browse, Preview, Download, Detokenize, Share | `READ VOLUME` | Read files from volume |
| Upload, Delete | `WRITE VOLUME` | Add/remove files in volume |
| Manage Permissions | `MANAGE` | Administer volume grants |

Ref: https://docs.databricks.com/aws/en/volumes/privileges

### Key Tables

```
permission_types (reference)      permission_assignments (fact)        volume_grants (UC tracking)
────────────────────────────    ─────────────────────────────────    ─────────────────────────────
| permission_type (PK1)    |←──| permission_type (FK)           |   | entity_id                 |
| action (PK2)             |←──| action (FK)                    |   | entity_name               |
| display_name             |   | entity_id                      |   | volume_fqn                |
| description              |   | entity_type (user/group/sp)    |   | uc_privilege              |
| sort_order               |   | entity_name                    |   |   (READ/WRITE/MANAGE)     |
| is_active                |   | resource_path (folder in vol)  |   | is_active                 |
────────────────────────────   | assigned_by                    |   | granted_at / revoked_at   |
                                | is_active                      |   ─────────────────────────────
                                ─────────────────────────────────
```

### Flow: Assign Permission

1. Admin selects: User + Folder + Actions (e.g., Browse, Upload)
2. App inserts into `permission_assignments` (folder-level, app-enforced)
3. App determines UC privileges needed: Browse→READ_VOLUME, Upload→WRITE_VOLUME
4. For each privilege, checks `volume_grants` — if not already granted:
   - Executes `GRANT READ VOLUME ON VOLUME catalog.schema.volume TO \`user@email\``
   - Inserts tracking row into `volume_grants`
5. User now has UC-level access to the volume + app-level folder restriction

### Flow: Revoke Permission

1. Admin revokes specific action for a user on a folder
2. App sets `is_active=FALSE` in `permission_assignments`
3. App checks: does this user still have ANY other active assignment needing that UC privilege?
4. If NO remaining references → executes `REVOKE READ VOLUME ON VOLUME ... FROM \`user\``
5. Marks `volume_grants` row inactive

### Why Two Levels?

- UC grants are at the **VOLUME** level (UC doesn't support folder-level ACLs)
- App logic handles **folder-level** restrictions (which subdirectories the user can see)
- `volume_grants` prevents duplicate GRANTs and ensures safe REVOKEs

### Query Patterns
```sql
-- Get all active permissions for a specific entity (with folder)
SELECT action, resource_path FROM permission_assignments
WHERE entity_id = '<id>' AND is_active = TRUE AND permission_type = 'files';

-- Check UC grants for an entity
SELECT uc_privilege, volume_fqn, granted_at FROM volume_grants
WHERE entity_name = 'user@email.com' AND is_active = TRUE;

-- Add a new permission category (no code change needed)
INSERT INTO permission_types VALUES ('clusters', 'create', 'Create Clusters', '...', 10, TRUE, now(), now());
```